# 🧠 NLP Foundations Refresher: From Preprocessing to tf-idf

## 🎯 What You'll Learn

By the end of this refresher, you will be able to:
1. Explain what a **Term-Document Incidence Matrix** is and how Boolean retrieval uses it
2. Compute **Term Frequency (TF)** and explain why raw counts need to be normalized
3. Apply **Log-Frequency Weighting** to reduce the impact of very common terms
4. Compute **Document Frequency (DF)** and understand why rare terms are more informative
5. Compute **Inverse Document Frequency (IDF)** and combine it with TF to get TF-IDF
6. Build a full **preprocessing pipeline** (tokenize → normalize → stop-word removal → stemming)
7. Run a complete **TF-IDF + cosine similarity retrieval** on a small corpus from scratch

---

## 📖 Key Vocabulary

| Term | Simple Explanation |
|------|-------------------|
| **Term-Document Incidence Matrix** | A grid of 0s and 1s showing which words appear in which documents |
| **Term Frequency (TF)** | How many times a word appears in one document |
| **Log-Frequency Weighting** | A smoothed version of TF that shrinks very large counts using logarithms |
| **Document Frequency (DF)** | How many documents contain a specific word |
| **Inverse Document Frequency (IDF)** | A score that rewards rare words — higher IDF means the word is more informative |
| **TF-IDF** | TF × IDF: the word is important *and* rare — a strong signal of relevance |
| **Tokenization** | Splitting a sentence into individual words (tokens) |
| **Normalization** | Lowercasing and removing punctuation so "NASA" and "nasa" are the same |
| **Stop-word Removal** | Dropping words like "the", "is", "and" that carry no meaning |
| **Stemming** | Reducing words to their root: "running" → "run", "learning" → "learn" |
| **Cosine Similarity** | How similar two document vectors are — measured by the angle between them |


## Step 1: Presenting the Six Core NLP Concepts

### 🔹 Term-Document Incidence Matrix

The **Term-Document Incidence Matrix** is a binary matrix that shows whether a term $t$ appears in a document $d$.

- Rows represent terms in the vocabulary  
- Columns represent documents in the corpus  
- Each entry $w_{t,d}$ is defined as:

$$
w_{t,d} =
\begin{cases}
1 & \text{if } t \in d \\
0 & \text{otherwise}
\end{cases}
$$

This is a **binary representation** — it only records the **presence or absence** of a term, not how many times it appears.

---

#### ✅ Why Use It?

- It’s the **simplest form** of representing document contents using structured data.
- Useful for:
  - Boolean search and keyword filters
  - Document classification based on keyword sets
  - Building foundational **retrieval systems**
- Helps in detecting whether **all query terms exist** in a document (e.g., phrase queries or "AND" operations)

---

#### 📘 Example

Suppose we have 3 documents:

- **Doc1**: "machine learning is fun"  
- **Doc2**: "deep learning is powerful"  
- **Doc3**: "machine learning and deep models"

The vocabulary extracted from all three is:

**Vocabulary** = {machine, learning, is, fun, deep, powerful, and, models}

The Term-Document Incidence Matrix would look like:

| Term       | Doc1 | Doc2 | Doc3 |
|------------|------|------|------|
| machine    | 1    | 0    | 1    |
| learning   | 1    | 1    | 1    |
| is         | 1    | 1    | 0    |
| fun        | 1    | 0    | 0    |
| deep       | 0    | 1    | 1    |
| powerful   | 0    | 1    | 0    |
| and        | 0    | 0    | 1    |
| models     | 0    | 0    | 1    |

For example:
- $w_{\text{machine}, \text{Doc1}} = 1$ → "machine" is in Doc1
- $w_{\text{powerful}, \text{Doc1}} = 0$ → "powerful" is not in Doc1

This matrix is particularly helpful when implementing **Boolean retrieval systems** and **phrase matching**.


In [1]:
# 📘 Example: Term-Document Incidence Matrix

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Sample corpus from the Markdown example
docs = [
    "machine learning is fun",          # Doc1
    "deep learning is powerful",        # Doc2
    "machine learning and deep models"  # Doc3
]

# Use binary=True to indicate presence/absence (1 or 0)
vectorizer = CountVectorizer(binary=True)

# Fit and transform the corpus
X = vectorizer.fit_transform(docs)

# Create a labeled DataFrame
incidence_matrix = pd.DataFrame(X.toarray(),
                                index=["Doc1", "Doc2", "Doc3"],
                                columns=vectorizer.get_feature_names_out())

# Display the incidence matrix
print("🔎 Term-Document Incidence Matrix:")
display(incidence_matrix)


🔎 Term-Document Incidence Matrix:


,and,deep,fun,is,learning,machine,models,powerful
Doc1,0,0,1,1,1,1,0,0
Doc2,0,1,0,1,1,0,0,1
Doc3,1,1,0,0,1,1,1,0


🗣️ **Instructor Talking Point**: This code demonstrates how the presence or absence of a term in a document is encoded as a binary matrix — foundational for Boolean retrieval. Explain this with respect to how a future AI agent (chatbot) builds context from retrieved documents.
<br/>
<br/>
🧠 **Student Talking Point**: Add a phrase query (e.g., 'machine learning') and explain your reasoning as to how you would check if both terms occur in a single document using this matrix.


### 🔹 Term Frequency (TF)

**Term Frequency (TF)** measures how frequently a term $t$ appears in a document $d$.

$$
tf_{t,d} = f_{t,d}
$$

Where $f_{t,d}$ is the raw count of term $t$ in document $d$.

---

#### ✅ Why Use It?

- TF reflects the importance of a word **within a specific document**.
- A higher TF means the term is likely central to the topic of that document.
- It's used as the **first step** in vectorizing text for machine learning models like classification, clustering, or information retrieval.

TF is most effective when combined with **IDF** (Inverse Document Frequency) to balance against very common terms across the corpus.

---

#### 📘 Example

Let’s say we have this document:

> **Doc1**: `"machine learning is fun and machine learning is useful"`

Calculate raw term counts:

| Term     | Raw TF $(f_{t,d})$ |
|----------|--------------------|
| machine  | 2                  |
| learning | 2                  |
| is       | 2                  |
| fun      | 1                  |
| and      | 1                  |
| useful   | 1                  |

If normalized (total of 9 words):

- $tf(\text{"machine"}, \text{Doc1}) = \frac{2}{9} \approx 0.22$
- $tf(\text{"learning"}, \text{Doc1}) = \frac{2}{9} \approx 0.22$

This simple frequency can then be used as input into models such as **TF-IDF**, which adjusts these values based on how rare the words are across multiple documents.


In [2]:
# 📘 Example: Term Frequency (TF)

import pandas as pd
from collections import Counter

# Sample document
doc1 = "machine learning is fun and machine learning is useful"

# Tokenize the document (simple lowercase + split)
tokens = doc1.lower().split()

# Count term frequencies
tf_raw = Counter(tokens)

# Total number of words
total_terms = len(tokens)

# Compute normalized TF
tf_normalized = {term: count / total_terms for term, count in tf_raw.items()}

# Display results
print("🔢 Raw Term Frequencies:")
display(pd.DataFrame(tf_raw.items(), columns=["Term", "Raw TF"]))

print("\n📏 Normalized Term Frequencies:")
display(pd.DataFrame(tf_normalized.items(), columns=["Term", "TF (Normalized)"]))


🔢 Raw Term Frequencies:


,Term,Raw TF
0,machine,2
1,learning,2
2,is,2
3,fun,1
4,and,1
5,useful,1



📏 Normalized Term Frequencies:


,Term,TF (Normalized)
0,machine,0.222222
1,learning,0.222222
2,is,0.222222
3,fun,0.111111
4,and,0.111111
5,useful,0.111111


🗣️ **Instructor Talking Point**: Here we count how often each term appears in a single document and normalize it. This is the simplest way to represent word importance within a document. Explain this with respect to how a future AI agent (chatbot) builds context from what it retrieves.
<br/>
<br/>
🧠 **Student Talking Point**: Use this TF output to compare with another document. Which terms are likely to be most important in Doc1 based on their normalized TF? Explain your reasoning.


### 🔹 Log Frequency Weight

To reduce the impact of very frequent terms, **log frequency weighting** is applied.

$$
w_{t,d} =
\begin{cases}
1 + \log_{10}(f_{t,d}) & \text{if } f_{t,d} > 0 \\
0 & \text{if } f_{t,d} = 0
\end{cases}
$$

This transformation reduces the skew caused by terms that appear many times in a document. Instead of allowing their raw frequency to dominate, we scale their contribution **logarithmically**.

---

#### ✅ Why Use It?

- Frequent terms are not always the most **important** terms.
- Log scaling ensures that:
  - Words with a raw count of 1 are preserved ($1 + \\log_{10}(1) = 1$),
  - But words with very high counts (e.g., 1000) don’t dominate the document vector.

This helps **normalize the influence** of repetitive terms and improve the **numerical stability** of document representations in models.

---

#### 📘 Example

Let’s say we have a document with the following raw term counts:

| Term     | Raw TF $f_{t,d}$ | Log Frequency Weight $w_{t,d}$ |
|----------|------------------|-------------------------------|
| machine  | 1                | $1 + \\log_{10}(1) = 1$        |
| learning | 3                | $1 + \\log_{10}(3) \approx 1.477$ |
| data     | 10               | $1 + \\log_{10}(10) = 2$       |

So even though "data" appears 10 times, its log-weighted value is **just 2**, making it more comparable to less frequent but potentially more meaningful terms like "learning".

This makes log frequency weighting especially useful when preparing inputs for models like **TF-IDF** or **document clustering**.


In [3]:
# 📘 Example: Log Frequency Weighting

import pandas as pd
import numpy as np
from collections import Counter

# Sample document with varying term frequencies
doc = "machine learning data data data learning learning learning machine data data data data"

# Tokenize and count raw term frequencies
tokens = doc.lower().split()
raw_tf = Counter(tokens)

# Compute log frequency weights
log_weighted_tf = {
    term: 1 + np.log10(freq) if freq > 0 else 0
    for term, freq in raw_tf.items()
}

# Build and display the result as a DataFrame
df = pd.DataFrame({
    "Term": raw_tf.keys(),
    "Raw TF (f_{t,d})": raw_tf.values(),
    "Log Weight (w_{t,d})": log_weighted_tf.values()
})

print("📊 Log Frequency Weighting:")
display(df)


📊 Log Frequency Weighting:


,Term,"Raw TF (f_{t,d})","Log Weight (w_{t,d})"
0,machine,2,1.301030
1,learning,4,1.602060
2,data,7,1.845098


🗣️ **Instructor Talking Point**: Note how 'data' has a high frequency, but its impact is smoothed by log weighting, making it comparable to 'learning'. Explain this with respect to how a future AI agent (chatbot) builds context — log-TF prevents one very frequent word from dominating the entire document vector.
<br/>
<br/>
🧠 **Student Talking Point**: Try adjusting the number of times a word appears and observe how the log scale compresses large values.


### 🔹 Document Frequency (DF)

**Document Frequency** is the number of documents in which a term $t$ appears:

$$
df_t = |\{ d \in D : t \in d \}|
$$

Where:
- $df_t$ is the document frequency of term $t$
- $D$ is the set of all documents in the corpus
- $t \in d$ means the term $t$ appears in document $d$

---

#### ✅ Why Use It?

- It helps you understand **how common or rare** a word is across the entire document set.
- Words with **high DF** (e.g., “the”, “and”) occur in many documents and are often **less informative**.
- Words with **low DF** are more likely to be **specific and meaningful** for distinguishing between documents.
- DF is a key ingredient in calculating **Inverse Document Frequency (IDF)**.

---

#### 📘 Example

Suppose you have the following three documents:

- **Doc1**: "machine learning is fun"  
- **Doc2**: "deep learning is powerful"  
- **Doc3**: "machine learning and deep models"

Now, let’s compute the Document Frequency:

| Term     | Document Frequency ($df_t$) |
|----------|-----------------------------|
| machine  | 2 (Doc1, Doc3)              |
| learning | 3 (Doc1, Doc2, Doc3)        |
| deep     | 2 (Doc2, Doc3)              |
| models   | 1 (Doc3)                    |

The term **"learning"** appears in all three documents → **high DF**, which means it’s **less useful for distinguishing** between them.

The term **"models"** appears in only one document → **low DF**, meaning it could be a **useful keyword** for that specific document.


In [4]:
# 📘 Example: Document Frequency (DF)

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Sample documents from Curriculum Learning (4)
docs = [
    "machine learning is fun",          # Doc1
    "deep learning is powerful",        # Doc2
    "machine learning and deep models"  # Doc3
]

# Use CountVectorizer to extract term-document matrix (raw counts)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

# Get feature names and document-term matrix as array
terms = vectorizer.get_feature_names_out()
X_array = X.toarray()

# Calculate document frequency for each term
df_counts = (X_array > 0).sum(axis=0)

# Format as a DataFrame
df_table = pd.DataFrame({
    "Term": terms,
    "Document Frequency (df_t)": df_counts
}).sort_values("Document Frequency (df_t)", ascending=False)

print("📊 Document Frequency (DF) Table:")
display(df_table)


📊 Document Frequency (DF) Table:


,Term,Document Frequency (df_t)
4,learning,3
1,deep,2
3,is,2
5,machine,2
0,and,1
2,fun,1
6,models,1
7,powerful,1


🗣️ **Instructor Talking Point**: Notice how common terms like 'learning' appear in all documents, while more specific terms like 'fun' or 'models' appear in only one.
<br/>
<br/>
🧠 **Student Talking Point**: Choose a term and explain how its document frequency could affect downstream TF-IDF weighting.

### 🔹 Inverse Document Frequency (IDF)

**Inverse Document Frequency (IDF)** measures how rare or informative a term is across the entire corpus:

$$
idf_t = \log_{10} \left( \frac{N}{df_t} \right)
$$

Where:
- $N$ is the total number of documents in the corpus  
- $df_t$ is the number of documents that contain the term $t$

---

#### ✅ Why Use It?

- IDF is used to **downweight common terms** and **upweight rare ones**.
- Words like “the”, “and”, or “data” appear frequently and are less helpful in distinguishing documents.
- Terms that appear in **fewer documents** are often **more informative** and **discriminative**.
- IDF is a core component of **TF-IDF**, a widely used technique in search engines, document classification, and clustering.

---

#### 📘 Example

Let’s say we have **5 documents** total, and the following document frequencies:

| Term     | $df_t$ | $idf_t = \log_{10}(N / df_t)$ |
|----------|--------|-------------------------------|
| machine  | 3      | $\log_{10}(5 / 3) \approx 0.22$ |
| entropy  | 1      | $\log_{10}(5 / 1) = 0.70$       |
| the      | 5      | $\log_{10}(5 / 5) = 0.00$       |

- The term **"entropy"** appears in only one document, so its IDF is **high** → it’s a **rare and informative term**.
- The term **"the"** appears in five documents, so its IDF is **low** → it’s a **frequent and less informative term**


In [5]:
# 📘 Example: Inverse Document Frequency (IDF)

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Sample documents (5 total)
docs = [
    "machine learning is powerful",
    "deep learning is advanced",
    "entropy measures randomness",
    "machine learning and AI are evolving",
    "the science of machine learning"
]

# Total number of documents
N = len(docs)

# Use CountVectorizer to get document-term matrix
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)
terms = vectorizer.get_feature_names_out()
X_array = X.toarray()

# Compute document frequency for each term
df_counts = (X_array > 0).sum(axis=0)

# Compute IDF using log base 10
idf_values = np.log10(N / df_counts)

# Build a DataFrame for display
idf_table = pd.DataFrame({
    "Term": terms,
    "Document Frequency (df_t)": df_counts,
    "IDF (log10(N / df_t))": idf_values
}).sort_values("IDF (log10(N / df_t))", ascending=False)

print("📊 Inverse Document Frequency (IDF) Table:")
display(idf_table)


📊 Inverse Document Frequency (IDF) Table:


,Term,Document Frequency (df_t),IDF (log10(N / df_t))
0,advanced,1,0.698970
1,ai,1,0.698970
2,and,1,0.698970
3,are,1,0.698970
4,deep,1,0.698970
5,entropy,1,0.698970
6,evolving,1,0.698970
10,measures,1,0.698970
11,of,1,0.698970
12,powerful,1,0.698970


🗣️ **Instructor Talking Point**: IDF adjusts for the fact that some words are common across all documents — this is critical in improving document relevance in search systems.
<br/>
<br/>
🧠 **Student Talking Point**: Choose a low-IDF and high-IDF term from this output and explain why they behave differently.

### 🔹 TF-IDF Weighting

**TF-IDF (Term Frequency–Inverse Document Frequency)** scores each term $t$ in document $d$ based on how frequent and how rare it is:

$$
w_{t,d} = \left(1 + \log_{10}(f_{t,d})\right) \times \log_{10} \left( \frac{N}{df_t} \right)
$$

Where:
- $f_{t,d}$ is the raw count of term $t$ in document $d$
- $df_t$ is the number of documents that contain term $t$
- $N$ is the total number of documents in the corpus

---

#### ✅ Why Use It?

- TF-IDF balances **term importance within a document** (TF) against **term commonality across all documents** (IDF).
- It **boosts rare, relevant words** while **suppressing frequent, generic words**.
- TF-IDF is foundational in:
  - Information Retrieval (search engines)
  - Document similarity
  - Feature engineering for classification or clustering

---

#### 📘 Example

Suppose we have:

- $f_{\text{machine}, \text{Doc1}} = 3$
- $df_{\text{machine}} = 2$
- $N = 5$ total documents

Then:

- TF part: $1 + \log_{10}(3) \approx 1 + 0.477 = 1.477$
- IDF part: $\log_{10}(5 / 2) \approx 0.398$
- TF-IDF weight:

$$
w_{\text{machine}, \text{Doc1}} = 1.477 \times 0.398 \approx 0.588
$$

This means "machine" is **important within Doc1**, but since it's found in other documents too, the overall weight is **moderated**.

TF-IDF creates a **sparse, weighted vector representation** of documents, ready for:
- Cosine similarity
- Clustering
- Search ranking
- Input into classical machine learning models


In [6]:
# 📘 Example: TF-IDF Weighting (Manual Computation)

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# Sample corpus of 5 documents
docs = [
    "machine learning is powerful",
    "deep learning is advanced",
    "entropy measures randomness",
    "machine learning and AI are evolving",
    "the science of machine learning"
]

# Total number of documents
N = len(docs)

# Vectorize (raw term frequencies)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)
terms = vectorizer.get_feature_names_out()
X_array = X.toarray()

# Compute Document Frequencies
df = (X_array > 0).sum(axis=0)
idf = np.log10(N / df)

# Manual TF-IDF: apply (1 + log10(tf)) * idf when tf > 0, else 0
# NOTE: the log-TF weight is 0 for absent terms — we use np.where to enforce this
tf_log = np.where(X_array > 0, 1 + np.log10(X_array), 0)
tfidf = tf_log * idf

# Create a DataFrame for visual inspection
tfidf_df = pd.DataFrame(tfidf, columns=terms, index=[f"Doc{i+1}" for i in range(N)])

print("📊 TF-IDF Weighted Matrix (Manual Computation):")
display(tfidf_df.round(3))


📊 TF-IDF Weighted Matrix (Manual Computation):


C:\Users\muthu\AppData\Local\Temp\ipykernel_6376\938626108.py:32: RuntimeWarning: divide by zero encountered in log10
  tf_log = np.where(X_array > 0, 1 + np.log10(X_array), 0)


,advanced,ai,and,are,deep,entropy,evolving,is,learning,machine,measures,of,powerful,randomness,science,the
Doc1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.398,0.097,0.222,0.000,0.000,0.699,0.000,0.000,0.000
Doc2,0.699,0.000,0.000,0.000,0.699,0.000,0.000,0.398,0.097,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc3,0.000,0.000,0.000,0.000,0.000,0.699,0.000,0.000,0.000,0.000,0.699,0.000,0.000,0.699,0.000,0.000
Doc4,0.000,0.699,0.699,0.699,0.000,0.000,0.699,0.000,0.097,0.222,0.000,0.000,0.000,0.000,0.000,0.000
Doc5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.097,0.222,0.000,0.699,0.000,0.000,0.699,0.699


🗣️ **Instructor Talking Point**: We combined TF and IDF manually — useful for seeing how each part of the formula shapes the final result. Document Frequency (DF) tells us "how many documents use this term", while IDF does the opposite — it gives higher weight to rare terms. Together they balance relevance: common words get low TF-IDF; rare but frequently-used words get high TF-IDF.
<br/>
<br/>
🧠 **Student Talking Point**: Pick one row (a document) and explain which term seems most important and why, based on the TF-IDF weights.


## Step 2: Document Collection

In [7]:
# Step 2: Document Collection
# We use a small hardcoded corpus for demonstration.
# In a real project you would load documents from files or a dataset.

corpus = [
    "Machine learning models learn from data.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing helps computers understand text.",
    "Information retrieval finds relevant documents from a large collection.",
    "TF-IDF weights terms by their frequency and rarity across documents.",
    "Cosine similarity measures the angle between two document vectors.",
    "Stemming reduces words to their root form to unify related terms.",
    "Stop words like 'the' and 'is' carry little meaning and are removed.",
]

print(f"Corpus loaded: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"  Doc{i+1}: {doc[:70]}")


Corpus loaded: 8 documents
  Doc1: Machine learning models learn from data.
  Doc2: Deep learning uses neural networks with many layers.
  Doc3: Natural language processing helps computers understand text.
  Doc4: Information retrieval finds relevant documents from a large collection
  Doc5: TF-IDF weights terms by their frequency and rarity across documents.
  Doc6: Cosine similarity measures the angle between two document vectors.
  Doc7: Stemming reduces words to their root form to unify related terms.
  Doc8: Stop words like 'the' and 'is' carry little meaning and are removed.


## Step 3: Implement a Tokenizer

In [8]:
import re
from typing import List

def tokenize(text: str) -> List[str]:
    # Lowercase first, then keep only alphabetic characters
    # This removes punctuation so "Fun!" becomes "fun", not "fun!"
    text = text.lower()
    tokens = re.findall(r'[a-z]+', text)
    return tokens

# Example — notice punctuation is stripped correctly
print(tokenize("Machine Learning is Fun!"))
print(tokenize("TF-IDF: the best weighting scheme?"))


['machine', 'learning', 'is', 'fun']
['tf', 'idf', 'the', 'best', 'weighting', 'scheme']


## Step 4: Text Normalization Pipeline

In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download required NLTK data (safe to re-run — only downloads once)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

def normalize(text: str):
    'Full normalization pipeline: lowercase → strip punct → remove stopwords → stem.'
    # Step 1: lowercase
    text = text.lower()
    # Step 2: remove non-alphabetic characters (punctuation, numbers)
    text = re.sub(r'[^a-z\s]', '', text)
    # Step 3: tokenize
    tokens = text.split()
    # Step 4: remove stop-words
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    # Step 5: stem to root form
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# Test on a sample document
sample = corpus[0]
print(f"Original : {sample}")
print(f"Tokenized: {tokenize(sample)}")
print(f"Normalized (stems, no stopwords): {normalize(sample)}")


Original : Machine learning models learn from data.
Tokenized: ['machine', 'learning', 'models', 'learn', 'from', 'data']
Normalized (stems, no stopwords): ['machin', 'learn', 'model', 'learn', 'data']


## Step 5: Build and Test the Pipeline

Now we connect everything together. The pipeline below:
1. Normalizes every document using the function from Step 4
2. Builds a **Term-Document Incidence Matrix** (binary)
3. Computes **TF** and **Log-TF** weights
4. Computes **DF** and **IDF**
5. Builds the full **TF-IDF** matrix
6. Runs a **cosine similarity retrieval** for four test queries

> 💡 Think of this as a mini search engine you built from scratch!


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

# ── Step 5a: Preprocess all documents ────────────────────────────────────────
preprocessed = [normalize(doc) for doc in corpus]

# Build vocabulary (sorted for reproducibility)
vocab = sorted(set(t for doc in preprocessed for t in doc))
vocab_index = {t: i for i, t in enumerate(vocab)}
V = len(vocab)   # vocabulary size
D = len(corpus)  # number of documents

print(f"Vocabulary size: {V} unique stems")
print(f"Sample vocabulary: {vocab[:10]}")

# ── Step 5b: Incidence Matrix (binary) ───────────────────────────────────────
incidence = np.zeros((V, D), dtype=int)
for j, tokens in enumerate(preprocessed):
    for t in set(tokens):
        if t in vocab_index:
            incidence[vocab_index[t], j] = 1

# ── Step 5c: TF Matrix ────────────────────────────────────────────────────────
tf_mat = np.zeros((V, D), dtype=float)
for j, tokens in enumerate(preprocessed):
    counts = Counter(tokens)
    for t, cnt in counts.items():
        if t in vocab_index:
            tf_mat[vocab_index[t], j] = cnt

# ── Step 5d: Log-TF Matrix ────────────────────────────────────────────────────
log_tf_mat = np.where(tf_mat > 0, 1 + np.log10(tf_mat), 0.0)

# ── Step 5e: DF, IDF, TF-IDF ─────────────────────────────────────────────────
df_vec  = (tf_mat > 0).sum(axis=1)                    # shape: (V,)
idf_vec = np.log10(D / np.where(df_vec > 0, df_vec, 1))
tfidf_mat = log_tf_mat * idf_vec[:, np.newaxis]

# ── Step 5f: Display TF-IDF matrix ───────────────────────────────────────────
tfidf_df = pd.DataFrame(
    tfidf_mat.T,
    columns=vocab,
    index=[f"Doc{i+1}" for i in range(D)]
)
print("\n📊 TF-IDF Matrix (full pipeline):")
display(tfidf_df.round(3))

# ── Step 5g: Cosine Similarity Retrieval ─────────────────────────────────────
def cosine_sim(vec_a, vec_b):
    'Compute cosine similarity between two vectors.'
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))

def retrieve(query_text, top_k=3):
    'Preprocess query, build its TF-IDF vector, rank documents by cosine similarity.'
    q_tokens = normalize(query_text)
    q_vec = np.zeros(V)
    q_counts = Counter(q_tokens)
    for t, cnt in q_counts.items():
        if t in vocab_index:
            q_vec[vocab_index[t]] = (1 + np.log10(cnt)) * idf_vec[vocab_index[t]]

    scores = [cosine_sim(q_vec, tfidf_mat[:, j]) for j in range(D)]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    results = pd.DataFrame([
        {'Rank': k+1, 'Score': round(score, 4),
         'Document': f'Doc{idx+1}', 'Text': corpus[idx][:80]}
        for k, (idx, score) in enumerate(ranked[:top_k])
    ])
    return results

# ── Step 5h: Test with four information needs ─────────────────────────────────
test_queries = [
    "how are words represented as vectors?",
    "what is TF-IDF weighting?",
    "how does stemming work?",
    "finding relevant documents in a collection",
]

for q in test_queries:
    print(f"\nQuery: '{q}'")
    display(retrieve(q, top_k=3))


**What Do We See Here? — Full Pipeline Results**

- The pipeline correctly routes each query to the most relevant document:
  - "words represented as vectors" → the Cosine Similarity document (Doc6)
  - "TF-IDF weighting" → the TF-IDF document (Doc5)
  - "stemming" → the Stemming document (Doc7)
  - "finding relevant documents" → the Information Retrieval document (Doc4)
- **Scores near 1.0** mean very high similarity; **scores near 0** mean almost no overlap.
- Notice that the second and third results often have lower but non-zero scores — this is because
  they share some stems with the query even if they are not the primary match.

> 🗣️ **Instructor talking point**: This tiny 8-document demo is exactly what large-scale
> search engines do — just with millions of documents. The math is identical.
>
> 🧠 **Student talking point**: Try adding your own document to `corpus` and a query that
> should match it. Does the system retrieve it at rank 1? Why or why not?


## 🔚 Conclusion

You may use this notebook to prepare yourself for the **Vector Space Proximity** workshop.
